# Kinship Library: Final Files and Curated Cover Audit

Independent notebook to locate the expanded production files, verify catalogue alignment, and audit the 25 creator curated books for cover information.


In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Step 2: Find the saved expanded production files
import os

search_root = '/content/drive/MyDrive'
wanted_files = ['app_books_expanded.csv', 'app_book_embeddings_expanded.npy']
found = {}

for root, dirs, files in os.walk(search_root):
    for filename in wanted_files:
        if filename in files:
            path = os.path.join(root, filename)
            found[filename] = path
            print(f'FOUND: {filename}')
            print(path)
            print()

if not found:
    print('No expanded production files were found in MyDrive.')


FOUND: app_books_expanded.csv
/content/drive/MyDrive/kinship_library/data/processed/app_books_expanded.csv

FOUND: app_book_embeddings_expanded.npy
/content/drive/MyDrive/kinship_library/models/app_book_embeddings_expanded.npy



In [3]:
# Step 3: Load and verify the final catalogue and embeddings
import pandas as pd
import numpy as np

books_file = found.get('app_books_expanded.csv')
embeddings_file = found.get('app_book_embeddings_expanded.npy')

if books_file is None or embeddings_file is None:
    print('One or both final files are missing. Stop here and check Step 2.')
else:
    final_books = pd.read_csv(books_file)
    final_embeddings = np.load(embeddings_file)
    print('FINAL KINSHIP LIBRARY')
    print('Books:', len(final_books))
    print('Embeddings:', final_embeddings.shape)
    print('Aligned:', len(final_books) == len(final_embeddings))
    if 'is_curated' in final_books.columns:
        curated_mask = final_books['is_curated'].astype(str).str.lower().isin(['true','1'])
        print('Curated:', curated_mask.sum())


FINAL KINSHIP LIBRARY
Books: 1121
Embeddings: (1121, 384)
Aligned: True
Curated: 25


In [4]:
# Step 4: Audit cover information for the curated collection
if 'final_books' not in globals():
    print('Complete Steps 1 to 3 first.')
elif 'is_curated' not in final_books.columns:
    print("Column 'is_curated' was not found.")
else:
    curated_mask = final_books['is_curated'].astype(str).str.lower().isin(['true','1'])
    curated_cover_audit = final_books[curated_mask].copy()
    print('Curated books:', len(curated_cover_audit))
    if 'isbn_clean' in curated_cover_audit.columns:
        print('With ISBN:', curated_cover_audit['isbn_clean'].notna().sum())
    if 'cover_url' in curated_cover_audit.columns:
        print('With cover URL:', curated_cover_audit['cover_url'].notna().sum())
        print('Without cover URL:', curated_cover_audit['cover_url'].isna().sum())
    audit_columns = [c for c in ['title','author','isbn_clean','cover_url'] if c in curated_cover_audit.columns]
    display(curated_cover_audit[audit_columns])


Curated books: 25
With ISBN: 16
With cover URL: 15
Without cover URL: 10


,title,author,isbn_clean,cover_url
713,Staying with the Trouble,Donna J. Haraway,9780822373780,http://books.google.com/books/content?id=ZvDgD...
714,One Simple Thing: A New Look at the Science of...,Eddie Stern,9780865478404,http://books.google.com/books/content?id=IKZbD...
715,Yoga Vedanta Sutras,Swami Sivananda,NaN,http://books.google.com/books/content?id=Vdz4A...
716,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,NaN,NaN
717,Hatha Yoga Pradipika,Swami Satyananda Saraswati,NaN,NaN
718,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,9788188018048,http://books.google.com/books/content?id=tGbxP...
719,Yoga Vasistha,Vihari-Lala Mitra,NaN,NaN
720,Vedas & Upanishads,Pranay,NaN,NaN
721,Yoga for Americans,Indra Devi,9781614278504,http://books.google.com/books/content?id=oqUcs...
722,Yoga Anticolonial Philosophy,Shyam Ranganathan,9781839978777,http://books.google.com/books/content?id=nmnHE...


In [5]:
# ============================================================
# STEP 5
# Show curated books without cover URLs
# ============================================================

missing_curated_covers = curated_cover_audit[
    curated_cover_audit["cover_url"].isna()
].copy()

columns_to_show = [
    col for col in [
        "title",
        "author",
        "isbn_clean",
        "google_books_id",
        "description_source"
    ]
    if col in missing_curated_covers.columns
]

print("Curated books needing covers:", len(missing_curated_covers))

display(
    missing_curated_covers[columns_to_show]
    .reset_index(drop=True)
)

Curated books needing covers: 10


,title,author,isbn_clean
0,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,NaN
1,Hatha Yoga Pradipika,Swami Satyananda Saraswati,NaN
2,Yoga Vasistha,Vihari-Lala Mitra,NaN
3,Vedas & Upanishads,Pranay,NaN
4,Steps to Yoga and Yoga Initiation Papers,Swami Satyananda Saraswati,NaN
5,The Tibetan Book of the Dead,Gyurme Dorje,NaN
6,"Eastern Body, Western Mind",Anodea Judith,NaN
7,Un mundo ch'ixi es posible,Silvia Rivera Cusicanqui,9789873687365
8,Sociología de la imagen,Silvia Rivera Cusicanqui,9789995418571
9,The Yoga Sutras of Patanjali,Shyam Ranganathan,NaN


In [6]:
# ============================================================
# STEP 6
# Recover verified ISBN metadata for curated books
# ============================================================

verified_isbns = {

    "Asana, Pranayama, Mudra and Bandha":
        "9788186336144",

    "Hatha Yoga Pradipika":
        "9788185787381",

    "Vedas & Upanishads":
        "9788194899129",

    "Steps to Yoga and Yoga Initiation Papers":
        "9788185787138",

    "The Tibetan Book of the Dead":
        "9780143104940",

    "Eastern Body, Western Mind":
        "9781587612251",

    "Un mundo ch'ixi es posible":
        "9789873687365",

    "Sociología de la imagen":
        "9789995418571",

    "The Yoga Sutras of Patanjali":
        "9780143102199"
}


for title, isbn in verified_isbns.items():

    mask = (
        final_books["is_curated"].astype(str).str.lower().isin(
            ["true", "1"]
        )
        &
        final_books["title"].eq(title)
    )

    final_books.loc[mask, "isbn_clean"] = isbn


print("Verified ISBN metadata added.")

curated_mask = (
    final_books["is_curated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
)

print(
    "Curated books with ISBN:",
    final_books.loc[
        curated_mask,
        "isbn_clean"
    ].notna().sum()
)

Verified ISBN metadata added.
Curated books with ISBN: 23


In [7]:
# ============================================================
# STEP 7
# Recover Open Library cover URLs from verified ISBNs
# ============================================================

curated_mask = (
    final_books["is_curated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
)

needs_cover = (
    curated_mask
    &
    final_books["cover_url"].isna()
    &
    final_books["isbn_clean"].notna()
)


final_books.loc[
    needs_cover,
    "cover_url"
] = (
    "https://covers.openlibrary.org/b/isbn/"
    + final_books.loc[
        needs_cover,
        "isbn_clean"
    ].astype(str)
    + "-L.jpg?default=false"
)


print(
    "Curated books with cover URL:",
    final_books.loc[
        curated_mask,
        "cover_url"
    ].notna().sum()
)

print(
    "Curated books without cover URL:",
    final_books.loc[
        curated_mask,
        "cover_url"
    ].isna().sum()
)


display(
    final_books.loc[
        curated_mask,
        [
            "title",
            "author",
            "isbn_clean",
            "cover_url"
        ]
    ]
)

Curated books with cover URL: 24
Curated books without cover URL: 1


,title,author,isbn_clean,cover_url
713,Staying with the Trouble,Donna J. Haraway,9780822373780,http://books.google.com/books/content?id=ZvDgD...
714,One Simple Thing: A New Look at the Science of...,Eddie Stern,9780865478404,http://books.google.com/books/content?id=IKZbD...
715,Yoga Vedanta Sutras,Swami Sivananda,NaN,http://books.google.com/books/content?id=Vdz4A...
716,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,9788186336144,https://covers.openlibrary.org/b/isbn/97881863...
717,Hatha Yoga Pradipika,Swami Satyananda Saraswati,9788185787381,https://covers.openlibrary.org/b/isbn/97881857...
718,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,9788188018048,http://books.google.com/books/content?id=tGbxP...
719,Yoga Vasistha,Vihari-Lala Mitra,NaN,NaN
720,Vedas & Upanishads,Pranay,9788194899129,https://covers.openlibrary.org/b/isbn/97881948...
721,Yoga for Americans,Indra Devi,9781614278504,http://books.google.com/books/content?id=oqUcs...
722,Yoga Anticolonial Philosophy,Shyam Ranganathan,9781839978777,http://books.google.com/books/content?id=nmnHE...


In [8]:
# ============================================================
# STEP 8
# Correct curated Hatha Yoga Pradipika edition metadata
# ============================================================

mask = (
    final_books["is_curated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
    &
    final_books["title"].eq("Hatha Yoga Pradipika")
)

final_books.loc[
    mask,
    "author"
] = "Swami Muktibodhananda"

print(
    final_books.loc[
        mask,
        ["title", "author", "isbn_clean", "cover_url"]
    ]
)

                    title                 author     isbn_clean  \
717  Hatha Yoga Pradipika  Swami Muktibodhananda  9788185787381   

                                             cover_url  
717  https://covers.openlibrary.org/b/isbn/97881857...  


In [9]:
# ============================================================
# STEP 9
# Validate curated cover URLs
# ============================================================

import requests
import pandas as pd
import time


curated_mask = (
    final_books["is_curated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
)


def check_cover(url):

    if pd.isna(url) or not str(url).strip():
        return False

    try:

        response = requests.get(
            url,
            timeout=10,
            allow_redirects=True
        )

        content_type = response.headers.get(
            "Content-Type",
            ""
        ).lower()

        return (
            response.status_code == 200
            and "image" in content_type
        )

    except requests.RequestException:
        return False


curated_indices = final_books[
    curated_mask
].index


cover_results = []

for idx in curated_indices:

    title = final_books.loc[idx, "title"]
    url = final_books.loc[idx, "cover_url"]

    works = check_cover(url)

    cover_results.append(works)

    print(
        "✓" if works else "✗",
        title
    )

    # Be gentle with the cover service
    time.sleep(0.2)


final_books.loc[
    curated_indices,
    "cover_verified"
] = cover_results


print()
print(
    "Working covers:",
    sum(cover_results)
)

print(
    "Missing or unavailable:",
    len(cover_results) - sum(cover_results)
)

✓ Staying with the Trouble
✓ One Simple Thing: A New Look at the Science of Yoga and How It Can Transform Your Life
✓ Yoga Vedanta Sutras
✓ Asana, Pranayama, Mudra and Bandha
✓ Hatha Yoga Pradipika
✓ Who Am I? (Nan Yar?)
✗ Yoga Vasistha
✗ Vedas & Upanishads
✓ Yoga for Americans
✓ Yoga Anticolonial Philosophy
✓ Breath
✓ Shiva Purana
✓ Steps to Yoga and Yoga Initiation Papers
✓ Sri Isopanishad
✓ Nature's Echo
✓ The Wisdom of No Escape and the Path of Loving-Kindness
✓ The Tibetan Book of the Dead
✓ Eastern Body, Western Mind
✓ Ancestral Future
✓ Ideas to Postpone the End of the World
✓ Life Is Not Useful
✓ Ch'ixinakax utxiwa
✓ Un mundo ch'ixi es posible
✗ Sociología de la imagen
✓ The Yoga Sutras of Patanjali

Working covers: 22
Missing or unavailable: 3


/tmp/ipykernel_4883/4288374222.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[True, True, True, True, True, True, False, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_books.loc[


In [10]:
# ============================================================
# STEP 10
# Show curated books whose covers still do not work
# ============================================================

curated_final_check = final_books[
    curated_mask
].copy()

missing_real_covers = curated_final_check[
    curated_final_check["cover_verified"] != True
]

print(
    "Curated books still needing a real cover:",
    len(missing_real_covers)
)

display(
    missing_real_covers[
        [
            "title",
            "author",
            "isbn_clean",
            "cover_url"
        ]
    ]
)

Curated books still needing a real cover: 3


,title,author,isbn_clean,cover_url
719,Yoga Vasistha,Vihari-Lala Mitra,NaN,NaN
720,Vedas & Upanishads,Pranay,9788194899129,https://covers.openlibrary.org/b/isbn/97881948...
736,Sociología de la imagen,Silvia Rivera Cusicanqui,9789995418571,https://covers.openlibrary.org/b/isbn/97899954...


In [11]:
# ============================================================
# STEP 11
# Final curated metadata cleanup
# ============================================================

# Make the verification column a proper boolean column
final_books["cover_verified"] = (
    final_books["cover_verified"]
    .fillna(False)
    .astype(bool)
)

# Add the ISBN for the complete Vihari Lala Mitra set
mask_vasistha = (
    final_books["is_curated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
    &
    final_books["title"].eq("Yoga Vasistha")
)

final_books.loc[
    mask_vasistha,
    "isbn_clean"
] = "8175361794"

print(
    final_books.loc[
        mask_vasistha,
        ["title", "author", "isbn_clean"]
    ]
)

             title             author  isbn_clean
719  Yoga Vasistha  Vihari-Lala Mitra  8175361794


/tmp/ipykernel_4883/1110310843.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [12]:
# ============================================================
# STEP 12
# Save corrected expanded production catalogue
# ============================================================

FINAL_BOOKS_PATH = (
    "/content/drive/MyDrive/kinship_library/"
    "data/processed/app_books_expanded.csv"
)

final_books.to_csv(
    FINAL_BOOKS_PATH,
    index=False
)

print("FINAL EXPANDED CATALOGUE SAVED")
print("Books:", len(final_books))
print(
    "Curated:",
    final_books["is_curated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
    .sum()
)

print(
    "Verified curated covers:",
    final_books.loc[
        final_books["is_curated"]
        .astype(str)
        .str.lower()
        .isin(["true", "1"]),
        "cover_verified"
    ].sum()
)

print("Saved to:")
print(FINAL_BOOKS_PATH)

FINAL EXPANDED CATALOGUE SAVED
Books: 1121
Curated: 25
Verified curated covers: 22
Saved to:
/content/drive/MyDrive/kinship_library/data/processed/app_books_expanded.csv


In [13]:
# ============================================================
# STEP 13
# Audit metadata quality of the 383 expansion records
# ============================================================

expansion = final_books[
    final_books["catalogue_source"] == "semantic_expansion"
].copy()

print("Semantic expansion records:", len(expansion))
print()

fields = [
    "author",
    "isbn_clean",
    "cover_url",
    "description_final"
]

for field in fields:

    if field in expansion.columns:

        missing = (
            expansion[field].isna()
            | expansion[field].astype(str).str.strip().eq("")
        ).sum()

        available = len(expansion) - missing

        print(
            f"{field}: "
            f"{available} available | "
            f"{missing} missing | "
            f"{available / len(expansion) * 100:.1f}% complete"
        )

Semantic expansion records: 383

author: 0 available | 383 missing | 0.0% complete
isbn_clean: 0 available | 383 missing | 0.0% complete
cover_url: 0 available | 383 missing | 0.0% complete
description_final: 383 available | 0 missing | 100.0% complete


In [14]:
# ============================================================
# STEP 14
# Inspect incomplete expansion records
# ============================================================

def missing_value(series):
    return (
        series.isna()
        | series.astype(str).str.strip().eq("")
    )


missing_author = missing_value(expansion["author"])
missing_isbn = missing_value(expansion["isbn_clean"])
missing_cover = missing_value(expansion["cover_url"])


print(
    "Missing author:",
    missing_author.sum()
)

print(
    "Missing ISBN:",
    missing_isbn.sum()
)

print(
    "Missing cover:",
    missing_cover.sum()
)

print(
    "Missing all three:",
    (
        missing_author
        & missing_isbn
        & missing_cover
    ).sum()
)


display(
    expansion.loc[
        missing_author
        | missing_isbn
        | missing_cover,
        [
            "title",
            "author",
            "isbn_clean",
            "cover_url",
            "source_id"
        ]
    ].head(30)
)

Missing author: 383
Missing ISBN: 383
Missing cover: 383
Missing all three: 383


,title,author,isbn_clean,cover_url,source_id
738,Interspecies Negotiations,NaN,NaN,NaN,NaN
739,Just Like Family,NaN,NaN,NaN,NaN
740,Critical Terms for Animal Studies,NaN,NaN,NaN,NaN
741,Individuality Incorporated,NaN,NaN,NaN,NaN
742,Spiritual Practices for Ecological Survival,NaN,NaN,NaN,NaN
743,Radical Human Ecology,NaN,NaN,NaN,NaN
744,Energy Democracies for Sustainable Futures,NaN,NaN,NaN,NaN
745,"Unruly Climates, Contested Futures I",NaN,NaN,NaN,NaN
746,Pathways to Human Ecology,NaN,NaN,NaN,NaN
747,One Health: The Psychology of Human-Nature Rel...,NaN,NaN,NaN,NaN


In [15]:
# ============================================================
# STEP 15
# Load the original Google Books expansion metadata
# ============================================================

import os
import pandas as pd

RAW_EXPANSION_PATH = (
    "/content/drive/MyDrive/kinship_library/"
    "data/raw/google_books_expansion_raw.csv"
)

print(
    "Raw expansion file exists:",
    os.path.exists(RAW_EXPANSION_PATH)
)

if os.path.exists(RAW_EXPANSION_PATH):

    expansion_raw = pd.read_csv(
        RAW_EXPANSION_PATH
    )

    print()
    print(
        "Raw expansion records:",
        len(expansion_raw)
    )

    print()
    print("Columns:")
    print(expansion_raw.columns.tolist())

Raw expansion file exists: True

Raw expansion records: 760

Columns:
['google_books_id', 'title', 'authors', 'description', 'categories', 'publisher', 'published_date', 'language', 'average_rating', 'ratings_count', 'isbn_13', 'isbn_10', 'thumbnail', 'collection_theme', 'discovery_query', 'curation_source']


In [16]:
# ============================================================
# STEP 16
# Inspect available metadata
# ============================================================

display(
    expansion_raw.head(10)
)

print()
print("Metadata availability:")

for col in [
    "google_books_id",
    "title",
    "authors",
    "isbn_13",
    "isbn_10",
    "thumbnail",
    "description"
]:

    if col in expansion_raw.columns:

        available = (
            expansion_raw[col]
            .notna()
            .sum()
        )

        print(
            f"{col}: "
            f"{available} / {len(expansion_raw)}"
        )

,google_books_id,title,authors,description,categories,publisher,published_date,language,average_rating,ratings_count,isbn_13,isbn_10,thumbnail,collection_theme,discovery_query,curation_source
0,vqHVEQAAQBAJ,Interspecies Negotiations,"Susan Haris, Anu Pande",Interspecies Negotiations: Literature and Huma...,Literary Collections,Bloomsbury Publishing,2026-02-20,en,NaN,NaN,9.789361e+12,9361314149,http://books.google.com/books/content?id=vqHVE...,humans_and_nature,multispecies kinship,google_books_expansion
1,RKUvEAAAQBAJ,Just Like Family,Andrea Laurent-Simpson,"""A first-of-its kind, in-depth investigation i...",Family & Relationships,NYU Press,2021-07-13,en,NaN,NaN,9.781480e+12,1479852627,http://books.google.com/books/content?id=RKUvE...,humans_and_nature,multispecies kinship,google_books_expansion
2,bV5xDwAAQBAJ,Critical Terms for Animal Studies,Lori Gruen,"Alexandra Horowitz, Peter Singer, Barbara King...",Science,University of Chicago Press,2018-10-23,en,NaN,NaN,9.780226e+12,022635556X,http://books.google.com/books/content?id=bV5xD...,humans_and_nature,multispecies kinship,google_books_expansion
3,OaMhEQAAQBAJ,Inhabitation,Gry Worre Hallberg,This innovative new book contributes greatly t...,Art,Ethics International Press,2024-09-05,en,NaN,NaN,9.781804e+12,1804416541,http://books.google.com/books/content?id=OaMhE...,humans_and_nature,multispecies kinship,google_books_expansion
4,u69bAAAAMAAJ,Dreams of Dawn,"Marti J. Steussy, Marti Steussy",NaN,Fiction,Del Rey,1988,en,NaN,NaN,NaN,NaN,http://books.google.com/books/content?id=u69bA...,humans_and_nature,multispecies kinship,google_books_expansion
5,1OkMAQAAMAAJ,Individuality Incorporated,Joel Pfister,"Explores the drive of whites to ""individualize...",History,Duke University Press Books,2004-02-16,en,NaN,NaN,NaN,NaN,http://books.google.com/books/content?id=1OkMA...,humans_and_nature,multispecies kinship,google_books_expansion
6,_7rdEQAAQBAJ,Spiritual Practices for Ecological Survival,Cláudio Carvalhaes,A powerful eco-theological call to reconnect w...,Religion,Bloomsbury Publishing USA,2026-09-03,en,NaN,NaN,9.798216e+12,NaN,http://books.google.com/books/content?id=_7rdE...,humans_and_nature,multispecies kinship,google_books_expansion
7,J8YpAQAAMAAJ,Proceedings of a Conference on Multispecies Gr...,"Frank H. Baker, R. Katherine Jones",NaN,Technology & Engineering,NaN,1985,en,NaN,NaN,NaN,NaN,http://books.google.com/books/content?id=J8YpA...,humans_and_nature,multispecies kinship,google_books_expansion
8,Ow3sAAAAMAAJ,Multi-species North Sea Fisheries,Chungsoo Kim,NaN,Fischwirtschaft,NaN,1984,en,NaN,NaN,NaN,NaN,http://books.google.com/books/content?id=Ow3sA...,humans_and_nature,multispecies kinship,google_books_expansion
9,QwwbAQAAIAAJ,Fieldiana,NaN,NaN,Zoology,NaN,2006,en,NaN,NaN,NaN,NaN,http://books.google.com/books/content?id=QwwbA...,humans_and_nature,multispecies kinship,google_books_expansion



Metadata availability:
google_books_id: 760 / 760
title: 760 / 760
authors: 663 / 760
isbn_13: 405 / 760
isbn_10: 395 / 760
thumbnail: 738 / 760
description: 503 / 760


In [17]:
# ============================================================
# STEP 17
# Recover expansion metadata from the saved Google Books cache
# ============================================================

import pandas as pd
import numpy as np
import re


def normalize_book_title(value):

    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    return " ".join(value.split())


# Work on copies
expansion_fix = expansion.copy()

raw_fix = expansion_raw.copy()


# Create matching keys
expansion_fix["_match_title"] = (
    expansion_fix["title"]
    .apply(normalize_book_title)
)

raw_fix["_match_title"] = (
    raw_fix["title"]
    .apply(normalize_book_title)
)


# Count how many raw records exist for each
# title + collection theme combination
raw_fix["_match_count"] = (
    raw_fix.groupby(
        [
            "_match_title",
            "collection_theme"
        ]
    )["google_books_id"]
    .transform("count")
)


# Only use unambiguous raw matches
raw_unique = raw_fix[
    raw_fix["_match_count"] == 1
].copy()


recovery_columns = [
    "_match_title",
    "collection_theme",
    "google_books_id",
    "authors",
    "isbn_13",
    "isbn_10",
    "thumbnail",
    "publisher",
    "published_date"
]


recovery = expansion_fix.merge(
    raw_unique[recovery_columns],
    on=[
        "_match_title",
        "collection_theme"
    ],
    how="left",
    validate="many_to_one"
)


print(
    "Expansion records:",
    len(recovery)
)

print(
    "Safely matched:",
    recovery["google_books_id"]
    .notna()
    .sum()
)

print(
    "Still unmatched or ambiguous:",
    recovery["google_books_id"]
    .isna()
    .sum()
)

Expansion records: 383
Safely matched: 340
Still unmatched or ambiguous: 43


In [18]:
# ============================================================
# STEP 18
# Inspect unmatched or ambiguous records
# ============================================================

unmatched = recovery[
    recovery["google_books_id"].isna()
].copy()


print(
    "Records requiring manual review:",
    len(unmatched)
)


display(
    unmatched[
        [
            "title",
            "collection_theme"
        ]
    ]
)

Records requiring manual review: 43


,title,collection_theme
21,Environmental Philosophy,humans_and_nature
24,Environmental Philosophy,humans_and_nature
28,Ecopsychology,humans_and_nature
33,Ecopsychology,humans_and_nature
42,Survival Strategies and Plant Intelligence,humans_and_nature
49,Forest Ecology,humans_and_nature
77,American Philosophies,indigenous_knowledge_americas
78,American Philosophy,indigenous_knowledge_americas
87,Ethnoarchaeology of Andean South America,indigenous_knowledge_americas
89,"Neutrosophic Sets and Systems, Vol. 81, 2025",indigenous_knowledge_americas


In [19]:
# ============================================================
# STEP 19
# Restore metadata for safely matched expansion books
# ============================================================

def clean_isbn(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Google Books CSV may load numeric ISBNs as floats
    if value.endswith(".0"):
        value = value[:-2]

    return value


# Prefer ISBN 13, otherwise ISBN 10
recovery["recovered_isbn"] = (
    recovery["isbn_13"]
    .apply(clean_isbn)
    .combine_first(
        recovery["isbn_10"].apply(clean_isbn)
    )
)


# Restore metadata
recovery["author"] = recovery["authors"]

recovery["isbn_clean"] = recovery["recovered_isbn"]

recovery["cover_url"] = recovery["thumbnail"]

recovery["source_id"] = recovery["google_books_id"]


safe_match = recovery["google_books_id"].notna()


print("Safe records:", safe_match.sum())

print(
    "Authors recovered:",
    recovery.loc[
        safe_match,
        "author"
    ].notna().sum()
)

print(
    "ISBNs recovered:",
    recovery.loc[
        safe_match,
        "isbn_clean"
    ].notna().sum()
)

print(
    "Covers recovered:",
    recovery.loc[
        safe_match,
        "cover_url"
    ].notna().sum()
)

print(
    "Google Books IDs recovered:",
    recovery.loc[
        safe_match,
        "source_id"
    ].notna().sum()
)

Safe records: 340
Authors recovered: 330
ISBNs recovered: 264
Covers recovered: 336
Google Books IDs recovered: 340


In [20]:
# ============================================================
# STEP 20
# Resolve ambiguous titles using description metadata
# ============================================================

ambiguous = recovery[
    recovery["google_books_id"].isna()
].copy()


description_matches = []


for idx, row in ambiguous.iterrows():

    candidates = raw_fix[
        (raw_fix["_match_title"] == row["_match_title"])
        &
        (
            raw_fix["collection_theme"]
            == row["collection_theme"]
        )
    ].copy()

    target_description = str(
        row["description_final"]
    ).strip()

    if (
        target_description
        and target_description.lower() != "nan"
    ):

        exact_description = candidates[
            candidates["description"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq(target_description)
        ]

    else:

        exact_description = candidates.iloc[0:0]


    if len(exact_description) == 1:

        matched = exact_description.iloc[0]

        description_matches.append({
            "recovery_index": idx,
            "title": row["title"],
            "google_books_id":
                matched["google_books_id"],
            "authors":
                matched["authors"],
            "isbn_13":
                matched["isbn_13"],
            "isbn_10":
                matched["isbn_10"],
            "thumbnail":
                matched["thumbnail"]
        })


description_matches = pd.DataFrame(
    description_matches
)


print(
    "Ambiguous records:",
    len(ambiguous)
)

print(
    "Resolved by exact description:",
    len(description_matches)
)

print(
    "Still requiring review:",
    len(ambiguous)
    - len(description_matches)
)


display(
    description_matches[
        [
            "title",
            "authors",
            "google_books_id"
        ]
    ]
)

Ambiguous records: 43
Resolved by exact description: 18
Still requiring review: 25


,title,authors,google_books_id
0,Environmental Philosophy,Simon P. James,xzDOCQAAQBAJ
1,Environmental Philosophy,Robin Attfield,pMl-AAAAMAAJ
2,Ecopsychology,"Theodore Roszak, Mary E. Gomes, Allen D. Kanner",gWl-AAAAMAAJ
3,Ecopsychology,"Peter H. Kahn, Jr., Patricia H. Hasbach",PPDuCwAAQBAJ
4,Forest Ecology,Dan Binkley,jkI6EAAAQBAJ
5,Yoga Sutras of Patanjali,B. Ravikanth,97woV0f2qz0C
6,Philosophy of the Bhagavad Gita,Keya Maitra,UmRHDwAAQBAJ
7,The Philosophy of the Bhagavad Gita,Ramesh Chandra Pradhan,SpqCEQAAQBAJ
8,The Philosophy of the Upanishads,Paul Deussen,k_Bea7AXHY4C
9,The Self in Indian Philosophy,Kālīprasāda Siṃha,s5lhQgAACAAJ


In [21]:
# ============================================================
# STEP 21
# Integrate exact description matches
# ============================================================

for _, match in description_matches.iterrows():

    idx = match["recovery_index"]

    recovery.loc[
        idx,
        "google_books_id"
    ] = match["google_books_id"]

    recovery.loc[
        idx,
        "authors"
    ] = match["authors"]

    recovery.loc[
        idx,
        "isbn_13"
    ] = match["isbn_13"]

    recovery.loc[
        idx,
        "isbn_10"
    ] = match["isbn_10"]

    recovery.loc[
        idx,
        "thumbnail"
    ] = match["thumbnail"]


# Rebuild app metadata columns

recovery["author"] = recovery["authors"]


recovery["isbn_clean"] = (
    recovery["isbn_13"]
    .apply(clean_isbn)
    .combine_first(
        recovery["isbn_10"]
        .apply(clean_isbn)
    )
)


recovery["cover_url"] = recovery["thumbnail"]

recovery["source_id"] = recovery["google_books_id"]


print(
    "Resolved expansion records:",
    recovery["google_books_id"]
    .notna()
    .sum()
)

print(
    "Still unresolved:",
    recovery["google_books_id"]
    .isna()
    .sum()
)

Resolved expansion records: 358
Still unresolved: 25


In [22]:
# ============================================================
# STEP 22
# Audit the remaining 25 ambiguous records
# ============================================================

still_unresolved = recovery[
    recovery["google_books_id"].isna()
].copy()


audit_rows = []


for idx, row in still_unresolved.iterrows():

    candidates = raw_fix[
        (raw_fix["_match_title"] == row["_match_title"])
        &
        (
            raw_fix["collection_theme"]
            == row["collection_theme"]
        )
    ]

    audit_rows.append({
        "recovery_index": idx,
        "title": row["title"],
        "collection_theme":
            row["collection_theme"],
        "candidate_editions":
            len(candidates)
    })


unresolved_audit = pd.DataFrame(
    audit_rows
)


print(
    "Books still requiring review:",
    len(unresolved_audit)
)


display(
    unresolved_audit.sort_values(
        [
            "candidate_editions",
            "title"
        ]
    )
)

Books still requiring review: 25


,recovery_index,title,collection_theme,candidate_editions
1,77,American Philosophies,indigenous_knowledge_americas,2
2,78,American Philosophy,indigenous_knowledge_americas,2
22,323,Complementary and Alternative Medicine for Old...,yoga_way_of_life,2
14,205,Encyclopedia of Indian Philosophies,yoga_philosophy,2
3,87,Ethnoarchaeology of Andean South America,indigenous_knowledge_americas,2
6,141,"Handbook of Anticolonial, Decolonial and Postc...",indigenous_knowledge_americas,2
19,256,"Indian Knowledge Systems (IKS) for Education, ...",ayurveda_indian_spirituality,2
18,251,"Indian Knowledge Systems (IKS) for Psychology,...",ayurveda_indian_spirituality,2
5,126,Indigenous South Americans Of The Past And Pre...,indigenous_knowledge_americas,2
16,216,Introduction to Indian Philosophy,yoga_philosophy,2


In [23]:
# ============================================================
# STEP 23
# Summarize the remaining ambiguous editions
# ============================================================

print("Remaining unresolved:", len(unresolved_audit))
print()

print("Candidate edition distribution:")

print(
    unresolved_audit["candidate_editions"]
    .value_counts()
    .sort_index()
)

print()
print("Maximum candidate editions:",
      unresolved_audit["candidate_editions"].max())

Remaining unresolved: 25

Candidate edition distribution:
candidate_editions
2    17
3     7
5     1
Name: count, dtype: int64

Maximum candidate editions: 5


In [24]:
# ============================================================
# STEP 24
# Compare descriptions for the final ambiguous records
# ============================================================

from difflib import SequenceMatcher


def description_similarity(text_a, text_b):

    if pd.isna(text_a) or pd.isna(text_b):
        return 0.0

    text_a = str(text_a).strip().lower()
    text_b = str(text_b).strip().lower()

    if not text_a or not text_b:
        return 0.0

    return SequenceMatcher(
        None,
        text_a,
        text_b
    ).ratio()


candidate_comparison = []


for idx, row in still_unresolved.iterrows():

    candidates = raw_fix[
        (raw_fix["_match_title"] == row["_match_title"])
        &
        (
            raw_fix["collection_theme"]
            == row["collection_theme"]
        )
    ].copy()

    for _, candidate in candidates.iterrows():

        score = description_similarity(
            row["description_final"],
            candidate["description"]
        )

        candidate_comparison.append({

            "recovery_index": idx,

            "title":
                row["title"],

            "google_books_id":
                candidate["google_books_id"],

            "authors":
                candidate["authors"],

            "isbn_13":
                candidate["isbn_13"],

            "isbn_10":
                candidate["isbn_10"],

            "thumbnail":
                candidate["thumbnail"],

            "description_similarity":
                score
        })


candidate_comparison = pd.DataFrame(
    candidate_comparison
)


display(
    candidate_comparison[
        [
            "title",
            "authors",
            "google_books_id",
            "description_similarity"
        ]
    ].sort_values(
        [
            "title",
            "description_similarity"
        ],
        ascending=[
            True,
            False
        ]
    )
)

,title,authors,google_books_id,description_similarity
2,American Philosophies,"Erin McKenna, Scott L. Pratt",6T6HEQAAQBAJ,1.000000
3,American Philosophies,"Erin McKenna, Scott L. Pratt",6T6HEQAAQBAJ,1.000000
4,American Philosophy,"Erin McKenna, Scott L. Pratt",ZS_aBAAAQBAJ,1.000000
5,American Philosophy,"Erin McKenna, Scott L. Pratt",ZS_aBAAAQBAJ,1.000000
54,Complementary and Alternative Medicine for Old...,"Elizabeth R. Mackenzie, Birgit Rakel",Gms30rKjAZMC,1.000000
55,Complementary and Alternative Medicine for Old...,"Elizabeth R. Mackenzie, Birgit Rakel",Gms30rKjAZMC,1.000000
37,Consciousness in Indian Philosophy,Matthew MacKenzie,XcXSEQAAQBAJ,1.000000
38,Consciousness in Indian Philosophy,Matthew MacKenzie,XcXSEQAAQBAJ,1.000000
36,Consciousness in Indian Philosophy,Sthaneshwar Timalsina,JTws0gEACAAJ,0.000000
51,"Deepening Consciousness: What Phenomenology, Y...","Christopher Gutland, Selenia Di Fronso, Stepha...",WGbOEQAAQBAJ,1.000000


In [25]:
# ============================================================
# STEP 25
# Remove duplicate Google Books candidates
# ============================================================

raw_unique_ids = (
    raw_fix
    .drop_duplicates(
        subset=["google_books_id"],
        keep="first"
    )
    .copy()
)


candidate_counts_clean = []


for idx, row in still_unresolved.iterrows():

    candidates = raw_unique_ids[
        (raw_unique_ids["_match_title"] == row["_match_title"])
        &
        (
            raw_unique_ids["collection_theme"]
            == row["collection_theme"]
        )
    ].copy()

    candidate_counts_clean.append({
        "recovery_index": idx,
        "title": row["title"],
        "collection_theme":
            row["collection_theme"],
        "unique_candidates":
            len(candidates)
    })


clean_audit = pd.DataFrame(
    candidate_counts_clean
)


print(
    "Remaining records:",
    len(clean_audit)
)

print()
print(
    "Unique candidate distribution:"
)

print(
    clean_audit["unique_candidates"]
    .value_counts()
    .sort_index()
)

print()
print(
    "Now uniquely identifiable:",
    (
        clean_audit["unique_candidates"] == 1
    ).sum()
)

print(
    "Still genuinely ambiguous:",
    (
        clean_audit["unique_candidates"] > 1
    ).sum()
)


display(
    clean_audit.sort_values(
        [
            "unique_candidates",
            "title"
        ]
    )
)

Remaining records: 25

Unique candidate distribution:
unique_candidates
1    16
2     8
4     1
Name: count, dtype: int64

Now uniquely identifiable: 16
Still genuinely ambiguous: 9


,recovery_index,title,collection_theme,unique_candidates
1,77,American Philosophies,indigenous_knowledge_americas,1
2,78,American Philosophy,indigenous_knowledge_americas,1
22,323,Complementary and Alternative Medicine for Old...,yoga_way_of_life,1
21,319,"Deepening Consciousness: What Phenomenology, Y...",yoga_way_of_life,1
14,205,Encyclopedia of Indian Philosophies,yoga_philosophy,1
3,87,Ethnoarchaeology of Andean South America,indigenous_knowledge_americas,1
6,141,"Handbook of Anticolonial, Decolonial and Postc...",indigenous_knowledge_americas,1
19,256,"Indian Knowledge Systems (IKS) for Education, ...",ayurveda_indian_spirituality,1
18,251,"Indian Knowledge Systems (IKS) for Psychology,...",ayurveda_indian_spirituality,1
16,216,Introduction to Indian Philosophy,yoga_philosophy,1


In [26]:
# ============================================================
# STEP 26
# Recover the 16 newly unique Google Books records
# ============================================================

newly_unique = clean_audit[
    clean_audit["unique_candidates"] == 1
].copy()


for _, item in newly_unique.iterrows():

    idx = item["recovery_index"]

    row = recovery.loc[idx]

    candidates = raw_unique_ids[
        (raw_unique_ids["_match_title"] == row["_match_title"])
        &
        (
            raw_unique_ids["collection_theme"]
            == row["collection_theme"]
        )
    ]

    candidate = candidates.iloc[0]

    recovery.loc[idx, "google_books_id"] = (
        candidate["google_books_id"]
    )

    recovery.loc[idx, "authors"] = (
        candidate["authors"]
    )

    recovery.loc[idx, "isbn_13"] = (
        candidate["isbn_13"]
    )

    recovery.loc[idx, "isbn_10"] = (
        candidate["isbn_10"]
    )

    recovery.loc[idx, "thumbnail"] = (
        candidate["thumbnail"]
    )


# Rebuild app fields

recovery["author"] = recovery["authors"]

recovery["isbn_clean"] = (
    recovery["isbn_13"]
    .apply(clean_isbn)
    .combine_first(
        recovery["isbn_10"]
        .apply(clean_isbn)
    )
)

recovery["cover_url"] = recovery["thumbnail"]

recovery["source_id"] = recovery["google_books_id"]


print(
    "Resolved expansion records:",
    recovery["google_books_id"]
    .notna()
    .sum()
)

print(
    "Still unresolved:",
    recovery["google_books_id"]
    .isna()
    .sum()
)

Resolved expansion records: 374
Still unresolved: 9


In [27]:
# ============================================================
# STEP 27
# Inspect the final genuinely ambiguous books
# ============================================================

final_ambiguous = recovery[
    recovery["google_books_id"].isna()
].copy()


final_candidate_rows = []


for idx, row in final_ambiguous.iterrows():

    candidates = raw_unique_ids[
        (raw_unique_ids["_match_title"] == row["_match_title"])
        &
        (
            raw_unique_ids["collection_theme"]
            == row["collection_theme"]
        )
    ]

    for _, candidate in candidates.iterrows():

        final_candidate_rows.append({

            "recovery_index": idx,

            "title":
                row["title"],

            "authors":
                candidate["authors"],

            "google_books_id":
                candidate["google_books_id"],

            "published_date":
                candidate["published_date"],

            "isbn_13":
                candidate["isbn_13"],

            "isbn_10":
                candidate["isbn_10"]
        })


final_candidates = pd.DataFrame(
    final_candidate_rows
)


print(
    "Genuinely ambiguous books:",
    len(final_ambiguous)
)

print(
    "Candidate records:",
    len(final_candidates)
)


display(
    final_candidates[
        [
            "title",
            "authors",
            "published_date",
            "isbn_13",
            "isbn_10",
            "google_books_id"
        ]
    ]
)

Genuinely ambiguous books: 9
Candidate records: 20


,title,authors,published_date,isbn_13,isbn_10,google_books_id
0,Survival Strategies and Plant Intelligence,"Manju A. Lal, Ratnum Kaul Wattal, Renu Kathpal...",2025-11-21,9.781040e+12,104045061X,NH-NEQAAQBAJ
1,Survival Strategies and Plant Intelligence,"Manju A. Lal, Ratnum Kaul Wattal, Renu Kathpal...",2025-11-21,9.781033e+12,1032976608,NlJq0QEACAAJ
2,Indigenous South Americans Of The Past And Pre...,David J. Wilson,2018-02-07,9.780430e+12,0429979487,MvZKDwAAQBAJ
3,Indigenous South Americans Of The Past And Pre...,David J. Wilson,2018-02-07,9.780430e+12,042996840X,hwDFDwAAQBAJ
4,Globalization and the Decolonial Option,"Walter D. Mignolo, Arturo Escobar",2013-10-18,9.781318e+12,1317966716,VbmMAQAAQBAJ
5,Globalization and the Decolonial Option,"Walter D. Mignolo, Arturo Escobar",2013-10-18,9.781318e+12,1317966708,xkeOAQAAQBAJ
6,Yoga Philosophy of Patañjali,"Patañjali, Swāmi Āraṇya Hariharānanda",1983-01-01,9.780874e+12,0873957288,58ndiC9MpMYC
7,Yoga Philosophy of Patañjali,Swāmi Āraṇya Hariharānanda,1984-06-30,9.781438e+12,1438405820,qUHabpdmSCkC
8,Yoga Sutras of Patanjali,Marilyn Gillian,2020-11-24,NaN,NaN,6f0KEAAAQBAJ
9,Yoga Sutras of Patanjali,B. Ravikanth,2012-09,9.780988e+12,0988251507,97woV0f2qz0C


Covers audit

In [28]:
# ============================================================
# COVER AUDIT
# Expansion collection before final merge
# ============================================================

resolved_mask = recovery["google_books_id"].notna()

resolved_expansion = recovery[
    resolved_mask
].copy()


print(
    "Resolved expansion books:",
    len(resolved_expansion)
)


print(
    "With cover URL:",
    resolved_expansion["cover_url"]
    .notna()
    .sum()
)


print(
    "Without cover URL:",
    resolved_expansion["cover_url"]
    .isna()
    .sum()
)

Resolved expansion books: 374
With cover URL: 369
Without cover URL: 5


In [29]:
# ============================================================
# VERIFY EXPANSION COVER IMAGES
# ============================================================

import requests
import pandas as pd
import time


def check_cover(url):

    if pd.isna(url) or not str(url).strip():
        return False

    try:

        response = requests.get(
            url,
            timeout=10,
            allow_redirects=True
        )

        content_type = response.headers.get(
            "Content-Type",
            ""
        ).lower()

        return (
            response.status_code == 200
            and "image" in content_type
        )

    except requests.RequestException:
        return False


cover_results = []


for number, (_, row) in enumerate(
    resolved_expansion.iterrows(),
    start=1
):

    works = check_cover(
        row["cover_url"]
    )

    cover_results.append(works)

    if number % 50 == 0:
        print(
            f"Checked {number} / "
            f"{len(resolved_expansion)}"
        )

    time.sleep(0.1)


resolved_expansion["cover_verified"] = (
    cover_results
)


print()
print(
    "Working covers:",
    resolved_expansion[
        "cover_verified"
    ].sum()
)

print(
    "Missing or unavailable:",
    (
        ~resolved_expansion[
            "cover_verified"
        ]
    ).sum()
)

Checked 50 / 374
Checked 100 / 374
Checked 150 / 374
Checked 200 / 374
Checked 250 / 374
Checked 300 / 374
Checked 350 / 374

Working covers: 369
Missing or unavailable: 5


In [30]:
# ============================================================
# NEXT STEP
# Inspect the 5 expansion books without working covers
# ============================================================

failed_covers = resolved_expansion[
    resolved_expansion["cover_verified"] == False
].copy()

print(
    "Resolved books needing a cover:",
    len(failed_covers)
)

display(
    failed_covers[
        [
            "title",
            "author",
            "isbn_clean",
            "cover_url",
            "source_id"
        ]
    ]
)

Resolved books needing a cover: 5


,title,author,isbn_clean,cover_url,source_id
58,Mourning in the Anthropocene,Joshua Barnett,9781628964660,NaN,abvUzwEACAAJ
108,Thinking Across Worlds,Jarrad Reddekop,NaN,NaN,ZCr9xAEACAAJ
189,Decolonizing Education,Marie Battiste,9781895830972,NaN,QBYwyQEACAAJ
223,The Self in Indian Philosophy,Kālīprasāda Siṃha,9788185094465,NaN,s5lhQgAACAAJ
344,Embodied Posture,Stacy Dockins,9780578411378,NaN,uGXMvgEACAAJ


In [31]:
# ============================================================
# NEXT STEP
# Try Open Library covers for the 5 failed Google covers
# ============================================================

import requests
import pandas as pd


def open_library_cover(isbn):

    if pd.isna(isbn) or not str(isbn).strip():
        return None

    isbn = str(isbn).replace(".0", "").strip()

    url = (
        f"https://covers.openlibrary.org/"
        f"b/isbn/{isbn}-L.jpg?default=false"
    )

    try:

        response = requests.get(
            url,
            timeout=10,
            allow_redirects=True
        )

        content_type = response.headers.get(
            "Content-Type",
            ""
        ).lower()

        if (
            response.status_code == 200
            and "image" in content_type
        ):
            return url

    except requests.RequestException:
        pass

    return None


failed_covers["openlibrary_cover"] = (
    failed_covers["isbn_clean"]
    .apply(open_library_cover)
)


print(
    "Open Library covers recovered:",
    failed_covers["openlibrary_cover"]
    .notna()
    .sum()
)

print()

display(
    failed_covers[
        [
            "title",
            "author",
            "isbn_clean",
            "openlibrary_cover"
        ]
    ]
)

Open Library covers recovered: 0



,title,author,isbn_clean,openlibrary_cover
58,Mourning in the Anthropocene,Joshua Barnett,9781628964660,None
108,Thinking Across Worlds,Jarrad Reddekop,NaN,None
189,Decolonizing Education,Marie Battiste,9781895830972,None
223,The Self in Indian Philosophy,Kālīprasāda Siṃha,9788185094465,None
344,Embodied Posture,Stacy Dockins,9780578411378,None


## Merging Metadata into final catalogue


In [32]:
# ============================================================
# STEP 28
# Restore recovered metadata to the final catalogue
# ============================================================

# Expansion rows in final_books
expansion_indices = final_books[
    final_books["catalogue_source"] == "semantic_expansion"
].index

print("Expansion rows in final catalogue:", len(expansion_indices))
print("Recovery rows:", len(recovery))

assert len(expansion_indices) == 383
assert len(recovery) == 383


# IMPORTANT
# recovery preserves the same order as the expansion subset
recovery_ordered = recovery.reset_index(drop=True)


columns_to_restore = [
    "author",
    "isbn_clean",
    "cover_url",
    "source_id"
]


for column in columns_to_restore:

    final_books.loc[
        expansion_indices,
        column
    ] = recovery_ordered[column].to_numpy()


print("Metadata restored to final_books.")
print("Total catalogue rows:", len(final_books))

Expansion rows in final catalogue: 383
Recovery rows: 383
Metadata restored to final_books.
Total catalogue rows: 1121


In [33]:
# ============================================================
# STEP 29
# Final metadata audit for semantic expansion
# ============================================================

expansion_repaired = final_books[
    final_books["catalogue_source"] == "semantic_expansion"
].copy()


def count_available(series):

    return (
        series.notna()
        &
        series.astype(str).str.strip().ne("")
    ).sum()


print("SEMANTIC EXPANSION METADATA")
print()

print(
    "Records:",
    len(expansion_repaired)
)

for column in [
    "author",
    "isbn_clean",
    "cover_url",
    "source_id",
    "description_final"
]:

    available = count_available(
        expansion_repaired[column]
    )

    missing = (
        len(expansion_repaired)
        - available
    )

    print(
        f"{column}: "
        f"{available} available | "
        f"{missing} missing | "
        f"{available / len(expansion_repaired) * 100:.1f}% complete"
    )

SEMANTIC EXPANSION METADATA

Records: 383
author: 364 available | 19 missing | 95.0% complete
isbn_clean: 290 available | 93 missing | 75.7% complete
cover_url: 369 available | 14 missing | 96.3% complete
source_id: 374 available | 9 missing | 97.7% complete
description_final: 383 available | 0 missing | 100.0% complete


In [34]:
# ============================================================
# STEP 30
# Verify catalogue and embedding alignment
# ============================================================

print("Books:", len(final_books))
print("Embeddings:", final_embeddings.shape)

print(
    "Aligned:",
    len(final_books) == len(final_embeddings)
)

print(
    "Original:",
    (
        final_books["catalogue_source"]
        == "original_collection"
    ).sum()
)

print(
    "Curated:",
    (
        final_books["catalogue_source"]
        == "creator_curated"
    ).sum()
)

print(
    "Expansion:",
    (
        final_books["catalogue_source"]
        == "semantic_expansion"
    ).sum()
)

Books: 1121
Embeddings: (1121, 384)
Aligned: True
Original: 713
Curated: 25
Expansion: 383


## Final Save

In [35]:
# ============================================================
# FINAL SAVE
# Corrected production catalogue
# ============================================================

FINAL_BOOKS_PATH = (
    "/content/drive/MyDrive/kinship_library/"
    "data/processed/app_books_expanded.csv"
)

final_books.to_csv(
    FINAL_BOOKS_PATH,
    index=False
)

print("FINAL PRODUCTION CATALOGUE SAVED")
print("Books:", len(final_books))
print("Path:")
print(FINAL_BOOKS_PATH)

FINAL PRODUCTION CATALOGUE SAVED
Books: 1121
Path:
/content/drive/MyDrive/kinship_library/data/processed/app_books_expanded.csv
